# 04 · Participant-level Knowledge Distillation

Run this after the No-KD baseline is frozen.

Stage 1 in this notebook is **Standard KD** using the exact same `ParticipantReLiMPNet` architecture.

Fixed protocol:
- TRAIN-107 locked participant teacher targets only
- audio/text teacher weights = 0.5 / 0.5
- temperature = 2.0
- KD weight = 0.5
- hard-label BCE retained
- threshold fixed at 0.5
- checkpoint selected on DEV-34 participant macro-F1
- DEV teacher outputs are not used for training
- TEST remains closed

Reliability-Aware KD is added only after Standard KD is frozen.


In [ ]:
from google.colab import drive,userdata
drive.mount('/content/drive')

from pathlib import Path
import base64,subprocess,sys,json
REPO='engrkhubabahmad/reliability-aware-depression-kd'
ROOT=Path('/content/reliability-aware-depression-kd')
token=userdata.get('GITHUB_TOKEN'); assert token,'Add GITHUB_TOKEN to Colab Secrets'
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git=['git','-c',f'http.extraHeader=AUTHORIZATION: basic {auth}']
if not (ROOT/'.git').exists(): subprocess.run(git+['clone',f'https://github.com/{REPO}.git',str(ROOT)],check=True)
subprocess.run(git+['-C',str(ROOT),'fetch','origin','main'],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','main'],check=True)
subprocess.run(git+['-C',str(ROOT),'pull','--ff-only','origin','main'],check=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-colab.txt')])
print('main:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


## 1. Paths and locked artifact preflight


In [ ]:
import pandas as pd,numpy as np

DATA=Path('/content/drive/MyDrive/DAIC_WOZ')
FEAT=DATA/'processed'/'participant_student'/'v1'
BASE=DATA/'experiments'/'students'/'participant_v1'/'no_kd'/'seed_103'
STD=DATA/'experiments'/'students'/'participant_v1'/'standard_kd'/'seed_103'

A=DATA/'experiments'/'ussd_audio_teacher_final'/'train_run4_crop_kd_targets.csv'
T=DATA/'experiments'/'idiap_text_teacher'/'train_text_kd_targets.csv'

for p in (FEAT/'feature_summary.json',BASE/'best.pt',BASE/'metrics.json',BASE/'model_complexity.json',A,T):
    assert p.exists(),p

a=pd.read_csv(A); t=pd.read_csv(T)
assert len(a)==107 and a.participant_id.nunique()==107
assert len(t)==107 and t.participant_id.nunique()==107
assert set(a.participant_id.astype(int))==set(t.participant_id.astype(int))
assert np.array_equal(
    a.sort_values('participant_id').label.to_numpy(int),
    t.sort_values('participant_id').label.to_numpy(int)
)
print('Locked TRAIN teacher alignment: OK')
print('No-KD baseline:',json.loads((BASE/'metrics.json').read_text())['dev34']['macro_f1'])
print('TEST remains closed.')


## 2. Standard KD

This trains from scratch with seed 103. It does **not** initialize from the No-KD checkpoint.


In [ ]:
%cd /content/reliability-aware-depression-kd

!python -u -m scripts.kd.participant_kd.train_standard \
  --features "{FEAT}" \
  --audio-train-targets "{A}" \
  --text-train-targets "{T}" \
  --baseline-output "{BASE}" \
  --output "{STD}" \
  --seed 103 \
  --batch-size 16 \
  --epochs 120 \
  --patience 20 \
  --lr 3e-4 \
  --weight-decay 1e-4 \
  --temperature 2.0 \
  --lambda-kd 0.5


## 3. Review Standard-KD result and direct student comparison


In [ ]:
std_metrics=json.loads((STD/'metrics.json').read_text())
std_complexity=json.loads((STD/'model_complexity.json').read_text())
std_protocol=json.loads((STD/'training_protocol.json').read_text())
comparison=pd.read_csv(STD/'student_comparison_dev34.csv')

print('STANDARD-KD DEV-34')
print(json.dumps(std_metrics['dev34'],indent=2))
print('\nCOMPLEXITY')
print(json.dumps(std_complexity,indent=2))
display(comparison)

assert std_protocol['same_architecture_as_no_kd'] is True
assert std_protocol['dev_teacher_targets_used_for_training'] is False
assert std_protocol['test_opened'] is False


## 4. Stop after Standard KD

Send the final **BEST STANDARD-KD DEV-34**, **COMPLEXITY**, and comparison table before running Reliability-Aware KD. We will freeze Standard KD first.
